In [1]:
import os
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import JsonOutputParser

# Set up NVIDIA API key (make sure to set this in your environment)
import dotenv
dotenv.load_dotenv()
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

In [2]:
# Define structured output schema using Pydantic
class Person(BaseModel):
    """Information about a person"""
    name: str = Field(description="The person's name")
    age: int = Field(description="The person's age")
    occupation: str = Field(description="The person's occupation")
    skills: List[str] = Field(description="List of skills")

class PersonInfo(BaseModel):
    """Container for person information"""
    person: Person = Field(description="Details about the person")
    summary: str = Field(description="A brief summary about the person")

In [10]:
# Initialize the NVIDIA chat model
chat_model = ChatNVIDIA(model="meta/llama-3.1-70b-instruct")

# Create the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert information extractor. Return only valid JSON that matches the requested schema. Do not add extra keys or commentary."),
    ("human", "Extract the person's information from the text below and return a structured JSON object with these fields: person.name, person.age, person.occupation, person.skills, and summary. Use only details explicitly stated in the text. If a field is not clearly mentioned, infer it only when it is obvious from context; otherwise leave it accurate and concise. The summary should be 1-2 sentences and should capture the person's identity, role, and notable abilities or experience.\n\nText:\n{text}")
])

# Initialize the JSON output parser with the schema
parser = JsonOutputParser(pydantic_object=PersonInfo)

# Create the LangChain pipeline using the | operator
chain = prompt | chat_model | parser

# Test the chain with sample input
sample_text = """
My name is Lakshya Sharma. I am 22 year old Student at sharda university. I am pursuing B.Tech in Computer Science and Engineering. I have a strong passion for coding and have experience with Python, Java, and C++. I have also worked on several projects related to machine learning and web development. In my free time, I enjoy participating in coding competitions and contributing to open-source projects.
"""

result = chain.invoke({"text": sample_text})
print("Extracted Person Information:")
print(result)
# save the result to json file
import json
with open("person_info.json", "w") as f:
    json.dump(result, f, indent=4)

Extracted Person Information:
{'person': {'name': 'Lakshya Sharma', 'age': 22, 'occupation': 'Student', 'skills': ['Python', 'Java', 'C++', 'Machine Learning', 'Web Development'], 'summary': 'Lakshya Sharma is a 22-year-old computer science student with a strong passion for coding and experience in machine learning and web development. He actively participates in coding competitions and contributes to open-source projects.'}}


In [9]:
# Subtask: parallel chaining with two independent branches
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert summarizer. Write concise, accurate summaries based only on the provided text. Do not invent details."),
    ("human", "Summarize the following person profile in 2-3 clear sentences. Include the person's name, age, occupation, key skills, and any notable experience or interests if present. Keep the tone factual and polished, and do not add information that is not explicitly supported by the text.\n\nText:\n{text}")
])

parallel_chain = RunnableParallel(
    extracted_person=chain,
    short_summary=analysis_prompt | chat_model | StrOutputParser(),
)

parallel_result = parallel_chain.invoke({"text": sample_text})

print("Parallel Chaining Result:")
print(json.dumps(parallel_result, indent=4))

Parallel Chaining Result:
{
    "extracted_person": {
        "person": {
            "name": "Lakshya Sharma",
            "age": null,
            "occupation": null,
            "skills": null,
            "summary": "Lakshya Sharma is an individual with no explicitly stated occupation or skills."
        }
    },
    "short_summary": "Lakshya Sharma is an individual with no specified age or occupation. No key skills or notable experience are mentioned in the provided information. The only detail provided is his name, Lakshya Sharma."
}
